In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path

cheaters_path = Path("../data/cheaters/cheaters.npy")
legit_path = Path("../data/legit/legit.npy")

cheaters = np.load(cheaters_path)
legit = np.load(legit_path)

X = np.concatenate([cheaters, legit], axis=0)
y = np.concatenate([
    np.ones(cheaters.shape[0], dtype=np.int64),
    np.zeros(legit.shape[0], dtype=np.int64),
])

channel_names = [
    "AttackerDeltaYaw",
    "AttackerDeltaPitch",
    "CrosshairToVictimYaw",
    "CrosshairToVictimPitch",
    "Firing",
]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"class 1 (cheaters): {int((y == 1).sum())}")
print(f"class 0 (legit):    {int((y == 0).sum())}")

X shape: (12000, 30, 192, 5)
y shape: (12000,)
class 1 (cheaters): 2000
class 0 (legit):    10000


In [2]:
# Clip unrealistic single-tick jumps in AttackerDeltaYaw / AttackerDeltaPitch.
# No human or aimbot moves >100 degrees in a single 1/64s tick, so values
# beyond that threshold are treated as artifacts (e.g. engagement-boundary
# resets seen in the EDA) rather than real aim movement.


def forward_fill_axis(arr, axis):
    """Forward-fill NaNs along `axis` (vectorized, no pandas)."""
    mask = np.isnan(arr)
    idx_shape = [1] * arr.ndim
    idx_shape[axis] = arr.shape[axis]
    idx = np.arange(arr.shape[axis]).reshape(idx_shape)
    idx = np.broadcast_to(idx, arr.shape).copy()
    idx[mask] = 0
    idx = np.maximum.accumulate(idx, axis=axis)
    return np.take_along_axis(arr, idx, axis=axis)


ARTIFACT_THRESHOLD = 100.0
yaw_idx = channel_names.index("AttackerDeltaYaw")
pitch_idx = channel_names.index("AttackerDeltaPitch")

total_clipped = 0
total_ticks_checked = 0

for ch_idx, ch_name in [(yaw_idx, "AttackerDeltaYaw"), (pitch_idx, "AttackerDeltaPitch")]:
    channel = X[..., ch_idx]
    mask = np.abs(channel) > ARTIFACT_THRESHOLD
    n_clipped = int(mask.sum())
    total_clipped += n_clipped
    total_ticks_checked += channel.size

    channel = channel.copy()
    channel[mask] = np.nan
    channel = forward_fill_axis(channel, axis=2)  # fill along tick axis, within each engagement
    X[..., ch_idx] = channel

    print(f"{ch_name}: clipped {n_clipped} / {channel.size} ticks "
          f"({n_clipped / channel.size * 100:.4f}%)")

pct_of_checked = total_clipped / total_ticks_checked * 100
print()
print(f"Total artifact values clipped: {total_clipped} "
      f"({pct_of_checked:.4f}% of yaw+pitch ticks checked)")

remaining_nan = np.isnan(X[..., yaw_idx]).sum() + np.isnan(X[..., pitch_idx]).sum()
print(f"Remaining NaN after forward-fill (leading-tick edge cases): {int(remaining_nan)}")

AttackerDeltaYaw: clipped 4745 / 69120000 ticks (0.0069%)


AttackerDeltaPitch: clipped 0 / 69120000 ticks (0.0000%)

Total artifact values clipped: 4745 (0.0034% of yaw+pitch ticks checked)


Remaining NaN after forward-fill (leading-tick edge cases): 47


In [3]:
feature_names = [
    "peak_yaw_delta",
    "peak_pitch_delta",
    "mean_yaw_delta",
    "snap_count",
    "min_cv_yaw",
    "min_cv_pitch",
    "cv_yaw_std",
    "cv_pitch_std",
    "fire_on_target_rate",
    "yaw_jerk",
    "engagement_firing_rate",
]


def engineer_features(player_tensor):
    """player_tensor: (30, 192, 5) -> 1D feature vector, averaged across engagements."""
    yaw = player_tensor[..., 0]
    pitch = player_tensor[..., 1]
    cv_yaw = player_tensor[..., 2]
    cv_pitch = player_tensor[..., 3]
    firing = player_tensor[..., 4]

    peak_yaw_delta = np.max(np.abs(yaw), axis=1)
    peak_pitch_delta = np.max(np.abs(pitch), axis=1)
    mean_yaw_delta = np.mean(np.abs(yaw), axis=1)
    snap_count = np.sum(np.abs(yaw) > 10, axis=1)
    min_cv_yaw = np.min(np.abs(cv_yaw), axis=1)
    min_cv_pitch = np.min(np.abs(cv_pitch), axis=1)
    cv_yaw_std = np.std(cv_yaw, axis=1)
    cv_pitch_std = np.std(cv_pitch, axis=1)

    firing_mask = firing == 1
    on_target = (np.abs(cv_yaw) < 5) & (np.abs(cv_pitch) < 5)
    n_on_target = np.sum(firing_mask & on_target, axis=1)
    n_firing = np.sum(firing_mask, axis=1)
    fire_on_target_rate = np.where(
        n_firing > 0, n_on_target / np.maximum(n_firing, 1), np.nan
    )

    yaw_jerk = np.std(np.diff(yaw, axis=1), axis=1)
    engagement_firing_rate = np.mean(firing == 1, axis=1)

    per_engagement = np.stack(
        [
            peak_yaw_delta,
            peak_pitch_delta,
            mean_yaw_delta,
            snap_count,
            min_cv_yaw,
            min_cv_pitch,
            cv_yaw_std,
            cv_pitch_std,
            fire_on_target_rate,
            yaw_jerk,
            engagement_firing_rate,
        ],
        axis=1,
    )  # (30, n_features)

    with np.errstate(invalid="ignore"):
        return np.nanmean(per_engagement, axis=0)


# sanity check on a single player
test_vec = engineer_features(X[0])
print(f"engineer_features output shape: {test_vec.shape}")
for name, val in zip(feature_names, test_vec):
    print(f"  {name}: {val:.4f}")

engineer_features output shape: (11,)
  peak_yaw_delta: 22.8261
  peak_pitch_delta: 2.7289
  mean_yaw_delta: 1.3649
  snap_count: 6.0000
  min_cv_yaw: 0.0921
  min_cv_pitch: 0.4523
  cv_yaw_std: 25.7589
  cv_pitch_std: 4.4717
  fire_on_target_rate: 0.6015
  yaw_jerk: 2.6609
  engagement_firing_rate: 0.0457


In [4]:
feature_rows = [engineer_features(X[i]) for i in range(X.shape[0])]
feature_matrix = np.array(feature_rows)

df = pd.DataFrame(feature_matrix, columns=feature_names)
df["label"] = y

print(f"df shape: {df.shape}")
print()
print(df.describe())
print()
print("NaN count per column:")
print(df.isna().sum())

df shape: (12000, 12)

       peak_yaw_delta  peak_pitch_delta  mean_yaw_delta    snap_count  \
count    12000.000000      12000.000000    12000.000000  12000.000000   
mean        16.616189          2.922465        1.028670      4.220811   
std          5.993441          1.286433        0.314748      2.233629   
min          2.357167          0.612233        0.244075      0.000000   
25%         12.338092          2.112250        0.811255      2.600000   
50%         15.806217          2.680717        0.992811      3.933333   
75%         20.106775          3.434683        1.209358      5.500000   
max         48.842967         49.329134        3.144039     20.400000   

         min_cv_yaw  min_cv_pitch    cv_yaw_std  cv_pitch_std  \
count  12000.000000  12000.000000  12000.000000  12000.000000   
mean       0.067477      0.174131     25.974661      4.471250   
std        0.075844      0.174726      7.230234      1.534780   
min        0.007167      0.002400      7.594263      1.2973

In [5]:
nan_counts_before = df.isna().sum()
affected_cols = nan_counts_before[nan_counts_before > 0]

if len(affected_cols) == 0:
    print("No NaN values found in any feature column.")
else:
    print("Rows affected per column (before fill):")
    for col, n in affected_cols.items():
        print(f"  {col}: {n} rows ({n / len(df) * 100:.2f}%)")
    for col in affected_cols.index:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  filled {col} NaNs with median={median_val:.4f}")

print()
print("NaN count per column (after fill):")
print(df.isna().sum())

No NaN values found in any feature column.

NaN count per column (after fill):
peak_yaw_delta            0
peak_pitch_delta          0
mean_yaw_delta            0
snap_count                0
min_cv_yaw                0
min_cv_pitch              0
cv_yaw_std                0
cv_pitch_std              0
fire_on_target_rate       0
yaw_jerk                  0
engagement_firing_rate    0
label                     0
dtype: int64


In [6]:
output_path = Path("../data/features.csv")
df.to_csv(output_path, index=False)

print(f"Saved features to {output_path.resolve()}")
print(f"Final shape: {df.shape}")

Saved features to D:\ARGUS\backend\data\features.csv
Final shape: (12000, 12)


In [7]:
features_df = pd.read_csv(Path("../data/features.csv"))

feature_cols = [c for c in features_df.columns if c != "label"]

cheater_group = features_df[features_df["label"] == 1]
legit_group = features_df[features_df["label"] == 0]

rows = []
for col in feature_cols:
    cheater_mean = cheater_group[col].mean()
    legit_mean = legit_group[col].mean()
    cheater_std = cheater_group[col].std()
    legit_std = legit_group[col].std()
    ratio = abs(cheater_mean - legit_mean) / ((cheater_std + legit_std) / 2)
    rows.append({
        "feature_name": col,
        "cheater_mean": cheater_mean,
        "legit_mean": legit_mean,
        "cheater_std": cheater_std,
        "legit_std": legit_std,
        "separation_ratio": ratio,
    })

separation_table = pd.DataFrame(rows).sort_values("separation_ratio", ascending=False)
separation_table = separation_table.reset_index(drop=True)

print("Quick group comparison (screening heuristic, NOT a formal statistical test):")
print(separation_table.to_string(index=False))

Quick group comparison (screening heuristic, NOT a formal statistical test):


          feature_name  cheater_mean  legit_mean  cheater_std  legit_std  separation_ratio
          min_cv_pitch      0.218004    0.165357     0.226487   0.160992          0.271742
          cv_pitch_std      4.275666    4.510367     1.455787   1.547201          0.156312
   fire_on_target_rate      0.642286    0.626127     0.129398   0.113783          0.132895
engagement_firing_rate      0.051171    0.052104     0.012898   0.010886          0.078440
            cv_yaw_std     25.618036   26.045986     7.166142   7.241233          0.059407
        peak_yaw_delta     16.325853   16.674256     5.937165   6.003242          0.058357
            snap_count      4.149683    4.235037     2.246007   2.230985          0.038130
        mean_yaw_delta      1.035873    1.027229     0.325284   0.312595          0.027100
            min_cv_yaw      0.068957    0.067181     0.054341   0.079449          0.026543
              yaw_jerk      1.580829    1.564813     0.673853   0.648854          0.02421

In [8]:
# NOTE: Cell 3's engineer_features() is left untouched (per instructions).
# compute_engagement_features() below computes the per-engagement values
# using the SAME formulas as Cell 3 -- the only difference is it returns
# the (30, 13) per-engagement table (11 features + 2 tick-index columns)
# instead of collapsing it with nanmean. Cell 9's engineer_features_v2()
# builds entirely on top of this function, so the per-engagement math is
# written once here and reused (not duplicated) by every v2 aggregation.


def compute_engagement_features(player_tensor):
    """player_tensor: (30, 192, 5) -> per-engagement DataFrame (30, 13).

    Same 11 formulas as Cell 3's engineer_features, plus 2 extra columns
    pinpointing the tick index of the extreme peak_yaw_delta / min_cv_yaw
    value within each engagement (for explainability traceability).
    """
    yaw = player_tensor[..., 0]
    pitch = player_tensor[..., 1]
    cv_yaw = player_tensor[..., 2]
    cv_pitch = player_tensor[..., 3]
    firing = player_tensor[..., 4]

    abs_yaw = np.abs(yaw)
    abs_cv_yaw = np.abs(cv_yaw)

    peak_yaw_delta = np.max(abs_yaw, axis=1)
    peak_yaw_delta_tick = np.argmax(abs_yaw, axis=1)
    peak_pitch_delta = np.max(np.abs(pitch), axis=1)
    mean_yaw_delta = np.mean(abs_yaw, axis=1)
    snap_count = np.sum(abs_yaw > 10, axis=1)
    min_cv_yaw = np.min(abs_cv_yaw, axis=1)
    min_cv_yaw_tick = np.argmin(abs_cv_yaw, axis=1)
    min_cv_pitch = np.min(np.abs(cv_pitch), axis=1)
    cv_yaw_std = np.std(cv_yaw, axis=1)
    cv_pitch_std = np.std(cv_pitch, axis=1)

    firing_mask = firing == 1
    on_target = (np.abs(cv_yaw) < 5) & (np.abs(cv_pitch) < 5)
    n_on_target = np.sum(firing_mask & on_target, axis=1)
    n_firing = np.sum(firing_mask, axis=1)
    fire_on_target_rate = np.where(
        n_firing > 0, n_on_target / np.maximum(n_firing, 1), np.nan
    )

    yaw_jerk = np.std(np.diff(yaw, axis=1), axis=1)
    engagement_firing_rate = np.mean(firing == 1, axis=1)

    return pd.DataFrame({
        "peak_yaw_delta": peak_yaw_delta,
        "peak_pitch_delta": peak_pitch_delta,
        "mean_yaw_delta": mean_yaw_delta,
        "snap_count": snap_count,
        "min_cv_yaw": min_cv_yaw,
        "min_cv_pitch": min_cv_pitch,
        "cv_yaw_std": cv_yaw_std,
        "cv_pitch_std": cv_pitch_std,
        "fire_on_target_rate": fire_on_target_rate,
        "yaw_jerk": yaw_jerk,
        "engagement_firing_rate": engagement_firing_rate,
        "peak_yaw_delta_tick": peak_yaw_delta_tick,
        "min_cv_yaw_tick": min_cv_yaw_tick,
    })


# sanity check on a single player
test_eng_df = compute_engagement_features(X[0])
print(f"compute_engagement_features output shape: {test_eng_df.shape}")
print(test_eng_df.head(3).to_string())

compute_engagement_features output shape: (30, 13)
   peak_yaw_delta  peak_pitch_delta  mean_yaw_delta  snap_count  min_cv_yaw  min_cv_pitch  cv_yaw_std  cv_pitch_std  fire_on_target_rate  yaw_jerk  engagement_firing_rate  peak_yaw_delta_tick  min_cv_yaw_tick
0        4.911000             0.758        0.335057           0       0.054         0.005    7.379697      1.380411                  1.0  0.551905                0.005208                   70              164
1       35.743999             1.208        1.718422           9       0.014         0.014   80.315071      3.912589                  1.0  3.820319                0.005208                  184              156
2       28.103001             3.686        0.795552           1       0.005         0.397   19.028812     12.138981                  0.5  2.717988                0.052083                   63              137


In [9]:
def engineer_features_v2(player_tensor):
    """player_tensor: (30, 192, 5) -> dict of aggregated (max/percentile-based)
    player-level features, with source engagement/tick traceability for the
    max/min-based features."""
    eng_df = compute_engagement_features(player_tensor)
    result = {}

    # peak_yaw_delta: MAX across engagements + source engagement + source tick
    peak_yaw_vals = eng_df["peak_yaw_delta"].values
    src_eng = int(np.nanargmax(peak_yaw_vals))
    result["peak_yaw_delta"] = peak_yaw_vals[src_eng]
    result["peak_yaw_delta_source_engagement"] = src_eng
    result["peak_yaw_delta_source_tick"] = int(eng_df["peak_yaw_delta_tick"].values[src_eng])

    # peak_pitch_delta: MAX across engagements (no source tracking requested)
    result["peak_pitch_delta"] = np.nanmax(eng_df["peak_pitch_delta"].values)

    # mean_yaw_delta: 95th percentile across engagements (percentile-based, no source)
    result["mean_yaw_delta"] = np.nanpercentile(eng_df["mean_yaw_delta"].values, 95)

    # snap_count: MAX across engagements + source engagement
    snap_vals = eng_df["snap_count"].values
    src_eng_snap = int(np.nanargmax(snap_vals))
    result["snap_count"] = snap_vals[src_eng_snap]
    result["snap_count_source_engagement"] = src_eng_snap

    # min_cv_yaw: MIN across engagements + source engagement + source tick
    min_cv_yaw_vals = eng_df["min_cv_yaw"].values
    src_eng_cvyaw = int(np.nanargmin(min_cv_yaw_vals))
    result["min_cv_yaw"] = min_cv_yaw_vals[src_eng_cvyaw]
    result["min_cv_yaw_source_engagement"] = src_eng_cvyaw
    result["min_cv_yaw_source_tick"] = int(eng_df["min_cv_yaw_tick"].values[src_eng_cvyaw])

    # min_cv_pitch: MIN across engagements + source engagement
    min_cv_pitch_vals = eng_df["min_cv_pitch"].values
    src_eng_cvpitch = int(np.nanargmin(min_cv_pitch_vals))
    result["min_cv_pitch"] = min_cv_pitch_vals[src_eng_cvpitch]
    result["min_cv_pitch_source_engagement"] = src_eng_cvpitch

    # cv_yaw_std: 5th percentile across engagements (percentile-based, no source)
    result["cv_yaw_std"] = np.nanpercentile(eng_df["cv_yaw_std"].values, 5)

    # cv_pitch_std: 5th percentile across engagements (percentile-based, no source)
    result["cv_pitch_std"] = np.nanpercentile(eng_df["cv_pitch_std"].values, 5)

    # fire_on_target_rate: 95th percentile across engagements (percentile-based, no source)
    result["fire_on_target_rate"] = np.nanpercentile(eng_df["fire_on_target_rate"].values, 95)

    # yaw_jerk: 5th percentile across engagements (percentile-based, no source)
    result["yaw_jerk"] = np.nanpercentile(eng_df["yaw_jerk"].values, 5)

    # engagement_firing_rate: 95th percentile across engagements (percentile-based, no source)
    result["engagement_firing_rate"] = np.nanpercentile(eng_df["engagement_firing_rate"].values, 95)

    return result


# sanity check on a single player
test_v2 = engineer_features_v2(X[0])
print("engineer_features_v2 sanity check (player 0):")
for k, v in test_v2.items():
    print(f"  {k}: {v}")

engineer_features_v2 sanity check (player 0):
  peak_yaw_delta: 40.095001220703125
  peak_yaw_delta_source_engagement: 21
  peak_yaw_delta_source_tick: 90
  peak_pitch_delta: 6.438000202178955
  mean_yaw_delta: 3.5766608715057373
  snap_count: 19
  snap_count_source_engagement: 21
  min_cv_yaw: 0.0
  min_cv_yaw_source_engagement: 29
  min_cv_yaw_source_tick: 133
  min_cv_pitch: 0.0
  min_cv_pitch_source_engagement: 20
  cv_yaw_std: 3.62804913520813
  cv_pitch_std: 1.1703100204467773
  fire_on_target_rate: 1.0
  yaw_jerk: 0.24900415539741516
  engagement_firing_rate: 0.10182291666666665


In [10]:
v2_rows = [engineer_features_v2(X[i]) for i in range(X.shape[0])]
df_v2 = pd.DataFrame(v2_rows)
df_v2["label"] = y

TICK_TO_SECONDS = 3.0 / 192  # ~3 second engagement window / 192 ticks

source_tick_cols = ["peak_yaw_delta_source_tick", "min_cv_yaw_source_tick"]
for col in source_tick_cols:
    real_time_col = col.replace("_source_tick", "_real_time_seconds")
    df_v2[real_time_col] = df_v2[col] * TICK_TO_SECONDS

output_path_v2 = Path("../data/features_v2.csv")
df_v2.to_csv(output_path_v2, index=False)

print(f"Saved v2 features to {output_path_v2.resolve()}")
print(f"df_v2 shape: {df_v2.shape}")
print(f"columns: {list(df_v2.columns)}")

Saved v2 features to D:\ARGUS\backend\data\features_v2.csv
df_v2 shape: (12000, 20)
columns: ['peak_yaw_delta', 'peak_yaw_delta_source_engagement', 'peak_yaw_delta_source_tick', 'peak_pitch_delta', 'mean_yaw_delta', 'snap_count', 'snap_count_source_engagement', 'min_cv_yaw', 'min_cv_yaw_source_engagement', 'min_cv_yaw_source_tick', 'min_cv_pitch', 'min_cv_pitch_source_engagement', 'cv_yaw_std', 'cv_pitch_std', 'fire_on_target_rate', 'yaw_jerk', 'engagement_firing_rate', 'label', 'peak_yaw_delta_real_time_seconds', 'min_cv_yaw_real_time_seconds']


In [11]:
features_v2_df = pd.read_csv(Path("../data/features_v2.csv"))

agg_feature_cols = [
    "peak_yaw_delta",
    "peak_pitch_delta",
    "mean_yaw_delta",
    "snap_count",
    "min_cv_yaw",
    "min_cv_pitch",
    "cv_yaw_std",
    "cv_pitch_std",
    "fire_on_target_rate",
    "yaw_jerk",
    "engagement_firing_rate",
]

cheater_group_v2 = features_v2_df[features_v2_df["label"] == 1]
legit_group_v2 = features_v2_df[features_v2_df["label"] == 0]

rows_v2 = []
for col in agg_feature_cols:
    cheater_mean = cheater_group_v2[col].mean()
    legit_mean = legit_group_v2[col].mean()
    cheater_std = cheater_group_v2[col].std()
    legit_std = legit_group_v2[col].std()
    ratio = abs(cheater_mean - legit_mean) / ((cheater_std + legit_std) / 2)
    rows_v2.append({
        "feature_name": col,
        "cheater_mean": cheater_mean,
        "legit_mean": legit_mean,
        "cheater_std": cheater_std,
        "legit_std": legit_std,
        "separation_ratio": ratio,
    })

separation_table_v2 = pd.DataFrame(rows_v2).sort_values("separation_ratio", ascending=False)
separation_table_v2 = separation_table_v2.reset_index(drop=True)

print("Quick group comparison on v2 (max/percentile-aggregated) features:")
print(separation_table_v2.to_string(index=False))

Quick group comparison on v2 (max/percentile-aggregated) features:
          feature_name  cheater_mean  legit_mean  cheater_std  legit_std  separation_ratio
          min_cv_pitch      0.001149    0.000598     0.002238   0.001530          0.292723
            min_cv_yaw      0.001673    0.001335     0.002164   0.001903          0.166407
          cv_pitch_std      1.435946    1.489807     0.627206   0.592352          0.088330
        peak_yaw_delta     40.582960   41.947886    16.751343  17.508113          0.079682
            cv_yaw_std      6.621903    6.485577     3.730044   3.635735          0.037016
      peak_pitch_delta      8.961659    9.105341     7.306854   6.572764          0.020704
engagement_firing_rate      0.097168    0.097539     0.026084   0.024207          0.014767
              yaw_jerk      0.391586    0.387524     0.272087   0.280949          0.014689
        mean_yaw_delta      2.165393    2.161431     0.739362   0.693716          0.005529
   fire_on_target_rate 

In [12]:
# Hybrid feature set: per prior findings (Cells 7 & 11), rate-like features
# (mean_yaw_delta, snap_count, cv_yaw_std, cv_pitch_std, fire_on_target_rate,
# yaw_jerk, engagement_firing_rate) do better with MEAN aggregation, while
# extreme-value features (peak_yaw_delta, peak_pitch_delta, min_cv_yaw,
# min_cv_pitch) do better with MAX/MIN aggregation. Built on top of
# compute_engagement_features() from Cell 8 -- no per-engagement math is
# re-derived here.


def engineer_features_hybrid(player_tensor):
    """player_tensor: (30, 192, 5) -> dict of hybrid-aggregated player-level
    features, with source engagement/tick traceability for peak_yaw_delta
    and min_cv_yaw (the two features flagged for explainability)."""
    eng_df = compute_engagement_features(player_tensor)
    result = {}

    # MAX aggregation, with source tracking
    peak_yaw_vals = eng_df["peak_yaw_delta"].values
    src_eng = int(np.nanargmax(peak_yaw_vals))
    result["peak_yaw_delta"] = peak_yaw_vals[src_eng]
    result["peak_yaw_delta_source_engagement"] = src_eng
    result["peak_yaw_delta_source_tick"] = int(eng_df["peak_yaw_delta_tick"].values[src_eng])

    # MAX aggregation, no source tracking needed
    result["peak_pitch_delta"] = np.nanmax(eng_df["peak_pitch_delta"].values)

    # MIN aggregation, with source tracking
    min_cv_yaw_vals = eng_df["min_cv_yaw"].values
    src_eng_cvyaw = int(np.nanargmin(min_cv_yaw_vals))
    result["min_cv_yaw"] = min_cv_yaw_vals[src_eng_cvyaw]
    result["min_cv_yaw_source_engagement"] = src_eng_cvyaw
    result["min_cv_yaw_source_tick"] = int(eng_df["min_cv_yaw_tick"].values[src_eng_cvyaw])

    # MIN aggregation, no source tracking needed
    result["min_cv_pitch"] = np.nanmin(eng_df["min_cv_pitch"].values)

    # MEAN aggregation (v1 style), no source tracking
    result["mean_yaw_delta"] = np.nanmean(eng_df["mean_yaw_delta"].values)
    result["snap_count"] = np.nanmean(eng_df["snap_count"].values)
    result["cv_yaw_std"] = np.nanmean(eng_df["cv_yaw_std"].values)
    result["cv_pitch_std"] = np.nanmean(eng_df["cv_pitch_std"].values)
    result["fire_on_target_rate"] = np.nanmean(eng_df["fire_on_target_rate"].values)
    result["yaw_jerk"] = np.nanmean(eng_df["yaw_jerk"].values)
    result["engagement_firing_rate"] = np.nanmean(eng_df["engagement_firing_rate"].values)

    return result


hybrid_rows = [engineer_features_hybrid(X[i]) for i in range(X.shape[0])]
df_hybrid = pd.DataFrame(hybrid_rows)
df_hybrid["label"] = y

TICK_TO_SECONDS = 3.0 / 192  # ~3 second engagement window / 192 ticks

hybrid_source_tick_cols = ["peak_yaw_delta_source_tick", "min_cv_yaw_source_tick"]
for col in hybrid_source_tick_cols:
    real_time_col = col.replace("_source_tick", "_real_time_seconds")
    df_hybrid[real_time_col] = df_hybrid[col] * TICK_TO_SECONDS

output_path_hybrid = Path("../data/features_hybrid.csv")
df_hybrid.to_csv(output_path_hybrid, index=False)

print(f"Saved hybrid features to {output_path_hybrid.resolve()}")
print(f"df_hybrid shape: {df_hybrid.shape}")
print(f"columns: {list(df_hybrid.columns)}")
print()
print("NaN count per column:")
print(df_hybrid.isna().sum())

Saved hybrid features to D:\ARGUS\backend\data\features_hybrid.csv
df_hybrid shape: (12000, 18)
columns: ['peak_yaw_delta', 'peak_yaw_delta_source_engagement', 'peak_yaw_delta_source_tick', 'peak_pitch_delta', 'min_cv_yaw', 'min_cv_yaw_source_engagement', 'min_cv_yaw_source_tick', 'min_cv_pitch', 'mean_yaw_delta', 'snap_count', 'cv_yaw_std', 'cv_pitch_std', 'fire_on_target_rate', 'yaw_jerk', 'engagement_firing_rate', 'label', 'peak_yaw_delta_real_time_seconds', 'min_cv_yaw_real_time_seconds']

NaN count per column:
peak_yaw_delta                      0
peak_yaw_delta_source_engagement    0
peak_yaw_delta_source_tick          0
peak_pitch_delta                    0
min_cv_yaw                          0
min_cv_yaw_source_engagement        0
min_cv_yaw_source_tick              0
min_cv_pitch                        0
mean_yaw_delta                      0
snap_count                          0
cv_yaw_std                          0
cv_pitch_std                        0
fire_on_target_rate    

In [13]:
hybrid_feature_cols = [
    "peak_yaw_delta",
    "peak_pitch_delta",
    "mean_yaw_delta",
    "snap_count",
    "min_cv_yaw",
    "min_cv_pitch",
    "cv_yaw_std",
    "cv_pitch_std",
    "fire_on_target_rate",
    "yaw_jerk",
    "engagement_firing_rate",
]

cheater_hybrid = df_hybrid[df_hybrid["label"] == 1]
legit_hybrid = df_hybrid[df_hybrid["label"] == 0]

ALPHA = 0.01
mw_rows = []
for col in hybrid_feature_cols:
    u_stat, p_val = stats.mannwhitneyu(
        cheater_hybrid[col], legit_hybrid[col], alternative="two-sided"
    )
    mw_rows.append({
        "feature_name": col,
        "u_statistic": u_stat,
        "p_value": p_val,
        "significant": p_val < ALPHA,
    })

mw_table = pd.DataFrame(mw_rows).sort_values("p_value", ascending=True).reset_index(drop=True)

print(f"Mann-Whitney U test (cheater vs legit), alpha = {ALPHA}:")
print(mw_table.to_string(index=False))
print()
n_significant = int(mw_table["significant"].sum())
print(f"{n_significant} / {len(mw_table)} features have p < {ALPHA}")

Mann-Whitney U test (cheater vs legit), alpha = 0.01:
          feature_name  u_statistic      p_value  significant
          min_cv_pitch   11789322.0 0.000000e+00         True
            min_cv_yaw   11110158.0 1.801412e-16         True
          cv_pitch_std    9085898.0 1.023734e-10         True
   fire_on_target_rate   10772306.5 4.739923e-08         True
        peak_yaw_delta    9555499.0 1.672474e-03         True
engagement_firing_rate    9622882.0 7.664204e-03         True
            cv_yaw_std    9664052.0 1.752953e-02        False
      peak_pitch_delta    9685463.0 2.614732e-02        False
            snap_count    9738744.0 6.470352e-02        False
              yaw_jerk   10137259.0 3.317855e-01        False
        mean_yaw_delta   10118648.0 4.015089e-01        False

6 / 11 features have p < 0.01


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

X_hybrid = df_hybrid[hybrid_feature_cols].values
y_hybrid = df_hybrid["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_hybrid, y_hybrid, test_size=0.2, stratify=y_hybrid, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models_dir = Path("../trained_models")
models_dir.mkdir(parents=True, exist_ok=True)
scaler_path = models_dir / "scaler.pkl"
joblib.dump(scaler, scaler_path)

splits_dir = Path("../data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
np.save(splits_dir / "X_train.npy", X_train)
np.save(splits_dir / "X_test.npy", X_test)
np.save(splits_dir / "y_train.npy", y_train)
np.save(splits_dir / "y_test.npy", y_test)
np.save(splits_dir / "X_train_scaled.npy", X_train_scaled)
np.save(splits_dir / "X_test_scaled.npy", X_test_scaled)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")
print(f"train cheater rate: {y_train.mean():.4f}, test cheater rate: {y_test.mean():.4f}")
print(f"scaler saved to {scaler_path.resolve()}")
print(f"splits saved to {splits_dir.resolve()}")

X_train shape: (9600, 11), X_test shape: (2400, 11)
y_train shape: (9600,), y_test shape: (2400,)
train cheater rate: 0.1667, test cheater rate: 0.1667
scaler saved to D:\ARGUS\backend\trained_models\scaler.pkl
splits saved to D:\ARGUS\backend\data\splits


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)

y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
auc_lr = roc_auc_score(y_test, y_proba_lr)

y_pred_lr = (y_proba_lr >= 0.5).astype(int)
precision_lr = precision_score(y_test, y_pred_lr, pos_label=1)
recall_lr = recall_score(y_test, y_pred_lr, pos_label=1)
f1_lr = f1_score(y_test, y_pred_lr, pos_label=1)

print("Logistic Regression (hybrid features, scaled, class_weight='balanced'):")
print(f"  AUC-ROC:   {auc_lr:.4f}")
print(f"  Precision (cheater class, threshold=0.5): {precision_lr:.4f}")
print(f"  Recall    (cheater class, threshold=0.5): {recall_lr:.4f}")
print(f"  F1        (cheater class, threshold=0.5): {f1_lr:.4f}")

Logistic Regression (hybrid features, scaled, class_weight='balanced'):
  AUC-ROC:   0.6359
  Precision (cheater class, threshold=0.5): 0.2410
  Recall    (cheater class, threshold=0.5): 0.5375
  F1        (cheater class, threshold=0.5): 0.3328


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)  # unscaled -- RF doesn't need feature scaling

y_proba_rf = rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_proba_rf)

importance_df = pd.DataFrame({
    "feature_name": hybrid_feature_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Random Forest (hybrid features, unscaled, class_weight='balanced'):")
print(f"  AUC-ROC: {auc_rf:.4f}")
print()
print("Feature importances (descending):")
print(importance_df.to_string(index=False))

Random Forest (hybrid features, unscaled, class_weight='balanced'):
  AUC-ROC: 0.6578

Feature importances (descending):
          feature_name  importance
   fire_on_target_rate    0.111878
engagement_firing_rate    0.109159
          cv_pitch_std    0.107248
        peak_yaw_delta    0.102443
      peak_pitch_delta    0.101137
              yaw_jerk    0.099334
            cv_yaw_std    0.097336
        mean_yaw_delta    0.096798
            snap_count    0.091987
          min_cv_pitch    0.045677
            min_cv_yaw    0.037003


In [17]:
# Hypothesis: aimbots may snap hard specifically in the first ~20 ticks of an
# engagement (target-acquisition window), then play normally afterward --
# whole-engagement features (Cells 3/8) may be diluting this early-window
# signal by averaging/maxing over all 192 ticks. These are new first-20-tick
# computations, not a reuse of compute_engagement_features()'s formulas.

EARLY_WINDOW_TICKS = 20


def compute_early_engagement_features(player_tensor):
    """player_tensor: (30, 192, 5) -> dict of 4 early-engagement (first 20
    ticks only) player-level features."""
    yaw = player_tensor[..., 0]
    pitch = player_tensor[..., 1]
    cv_yaw = player_tensor[..., 2]

    early_yaw = yaw[:, :EARLY_WINDOW_TICKS]        # (30, 20)
    early_pitch = pitch[:, :EARLY_WINDOW_TICKS]
    early_cv_yaw = cv_yaw[:, :EARLY_WINDOW_TICKS]

    abs_early_yaw = np.abs(early_yaw)
    abs_early_pitch = np.abs(early_pitch)
    abs_early_cv_yaw = np.abs(early_cv_yaw)

    per_eng_peak_yaw = np.max(abs_early_yaw, axis=1)      # (30,)
    per_eng_peak_pitch = np.max(abs_early_pitch, axis=1)
    per_eng_min_cv_yaw = np.min(abs_early_cv_yaw, axis=1)
    per_eng_lock_on = np.any(abs_early_cv_yaw < 5, axis=1)  # bool (30,)

    return {
        "early_peak_yaw_delta": np.nanmax(per_eng_peak_yaw),
        "early_peak_pitch_delta": np.nanmax(per_eng_peak_pitch),
        "early_min_cv_yaw": np.nanmin(per_eng_min_cv_yaw),
        "early_lock_on_rate": np.mean(per_eng_lock_on),
    }


# sanity check on a single player
test_early = compute_early_engagement_features(X[0])
print("compute_early_engagement_features sanity check (player 0):")
for k, v in test_early.items():
    print(f"  {k}: {v}")

compute_early_engagement_features sanity check (player 0):
  early_peak_yaw_delta: 19.80299949645996
  early_peak_pitch_delta: 4.735000133514404
  early_min_cv_yaw: 0.02500000037252903
  early_lock_on_rate: 0.2


In [18]:
early_rows = [compute_early_engagement_features(X[i]) for i in range(X.shape[0])]
df_early = pd.DataFrame(early_rows)
df_early["label"] = y

hybrid_csv_df = pd.read_csv(Path("../data/features_hybrid.csv"))
assert (hybrid_csv_df["label"].values == df_early["label"].values).all(), \
    "row order mismatch between features_hybrid.csv and freshly computed early features"

df_early_no_label = df_early.drop(columns=["label"])
df_combined = pd.concat([hybrid_csv_df, df_early_no_label], axis=1)

output_path_combined = Path("../data/features_hybrid_plus_early.csv")
df_combined.to_csv(output_path_combined, index=False)

early_feature_cols = [
    "early_peak_yaw_delta",
    "early_peak_pitch_delta",
    "early_min_cv_yaw",
    "early_lock_on_rate",
]

print(f"Saved combined features to {output_path_combined.resolve()}")
print(f"df_combined shape: {df_combined.shape}")
print(f"columns: {list(df_combined.columns)}")
print(f"modeling feature set: 11 hybrid + {len(early_feature_cols)} early = "
      f"{len(hybrid_feature_cols) + len(early_feature_cols)} features")
print()
print("NaN count in new early columns:")
print(df_combined[early_feature_cols].isna().sum())

Saved combined features to D:\ARGUS\backend\data\features_hybrid_plus_early.csv
df_combined shape: (12000, 22)
columns: ['peak_yaw_delta', 'peak_yaw_delta_source_engagement', 'peak_yaw_delta_source_tick', 'peak_pitch_delta', 'min_cv_yaw', 'min_cv_yaw_source_engagement', 'min_cv_yaw_source_tick', 'min_cv_pitch', 'mean_yaw_delta', 'snap_count', 'cv_yaw_std', 'cv_pitch_std', 'fire_on_target_rate', 'yaw_jerk', 'engagement_firing_rate', 'label', 'peak_yaw_delta_real_time_seconds', 'min_cv_yaw_real_time_seconds', 'early_peak_yaw_delta', 'early_peak_pitch_delta', 'early_min_cv_yaw', 'early_lock_on_rate']
modeling feature set: 11 hybrid + 4 early = 15 features

NaN count in new early columns:
early_peak_yaw_delta      0
early_peak_pitch_delta    0
early_min_cv_yaw          0
early_lock_on_rate        0
dtype: int64


In [19]:
cheater_combined = df_combined[df_combined["label"] == 1]
legit_combined = df_combined[df_combined["label"] == 0]

mw_early_rows = []
for col in early_feature_cols:
    u_stat, p_val = stats.mannwhitneyu(
        cheater_combined[col], legit_combined[col], alternative="two-sided"
    )
    mw_early_rows.append({
        "feature_name": col,
        "u_statistic": u_stat,
        "p_value": p_val,
        "significant": p_val < ALPHA,
    })

mw_early_table = pd.DataFrame(mw_early_rows).sort_values("p_value", ascending=True).reset_index(drop=True)

print(f"Mann-Whitney U test on the 4 new early-engagement features, alpha = {ALPHA}:")
print(mw_early_table.to_string(index=False))
print()
n_significant_early = int(mw_early_table["significant"].sum())
print(f"{n_significant_early} / {len(mw_early_table)} early features have p < {ALPHA}")

Mann-Whitney U test on the 4 new early-engagement features, alpha = 0.01:
          feature_name  u_statistic      p_value  significant
      early_min_cv_yaw   10731610.0 2.290082e-07         True
early_peak_pitch_delta    9704931.0 3.694568e-02        False
    early_lock_on_rate    9935999.5 6.498805e-01        False
  early_peak_yaw_delta    9953268.0 7.410767e-01        False

1 / 4 early features have p < 0.01


In [20]:
feature_cols_15 = hybrid_feature_cols + early_feature_cols

X_15 = df_combined[feature_cols_15].values
y_15 = df_combined["label"].values

X_train_15, X_test_15, y_train_15, y_test_15 = train_test_split(
    X_15, y_15, test_size=0.2, stratify=y_15, random_state=42
)

rf_15 = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_15.fit(X_train_15, y_train_15)

y_proba_rf15 = rf_15.predict_proba(X_test_15)[:, 1]
auc_rf15 = roc_auc_score(y_test_15, y_proba_rf15)

delta_15_vs_11 = auc_rf15 - auc_rf

print(f"Random Forest, 11 hybrid features only (Cell 16): AUC-ROC = {auc_rf:.4f}")
print(f"Random Forest, 15 features (hybrid + early):      AUC-ROC = {auc_rf15:.4f}")
print(f"Delta (15-feature - 11-feature): {delta_15_vs_11:+.4f}")

Random Forest, 11 hybrid features only (Cell 16): AUC-ROC = 0.6578
Random Forest, 15 features (hybrid + early):      AUC-ROC = 0.6436
Delta (15-feature - 11-feature): -0.0143


In [21]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=200, class_weight="balanced", random_state=42, verbosity=-1
)
lgbm.fit(X_train_15, y_train_15)

y_proba_lgbm = lgbm.predict_proba(X_test_15)[:, 1]
auc_lgbm = roc_auc_score(y_test_15, y_proba_lgbm)

delta_lgbm_vs_rf11 = auc_lgbm - auc_rf
delta_lgbm_vs_rf15 = auc_lgbm - auc_rf15

importance_lgbm_df = pd.DataFrame({
    "feature_name": feature_cols_15,
    "importance": lgbm.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(f"LightGBM, 15 features (hybrid + early): AUC-ROC = {auc_lgbm:.4f}")
print(f"  vs 11-feature RF (Cell 16, {auc_rf:.4f}):  delta = {delta_lgbm_vs_rf11:+.4f}")
print(f"  vs 15-feature RF (Cell 20, {auc_rf15:.4f}): delta = {delta_lgbm_vs_rf15:+.4f}")
print()
print("LightGBM feature importances (descending):")
print(importance_lgbm_df.to_string(index=False))

LightGBM, 15 features (hybrid + early): AUC-ROC = 0.6170
  vs 11-feature RF (Cell 16, 0.6578):  delta = -0.0409
  vs 15-feature RF (Cell 20, 0.6436): delta = -0.0266

LightGBM feature importances (descending):
          feature_name  importance
        peak_yaw_delta         525
   fire_on_target_rate         510
engagement_firing_rate         493
      early_min_cv_yaw         479
      peak_pitch_delta         468
          cv_pitch_std         463
early_peak_pitch_delta         463
            snap_count         452
  early_peak_yaw_delta         445
              yaw_jerk         439
        mean_yaw_delta         421
            cv_yaw_std         396
    early_lock_on_rate         211
          min_cv_pitch         126
            min_cv_yaw         109


In [22]:
# Engagement-level validation: does scoring each of the 360,000 engagements
# independently (unsupervised) and aggregating per-player beat the ~0.66 AUC
# ceiling we hit with player-level averaged features? Reuses
# compute_engagement_features() from Cell 8 -- no per-engagement math is
# re-derived here.

N_PLAYERS, N_ENGAGEMENTS = X.shape[0], X.shape[1]

engagement_feature_matrix = np.empty((N_PLAYERS * N_ENGAGEMENTS, len(feature_names)), dtype=np.float64)
player_idx_arr = np.empty(N_PLAYERS * N_ENGAGEMENTS, dtype=np.int64)
engagement_idx_arr = np.empty(N_PLAYERS * N_ENGAGEMENTS, dtype=np.int64)

for p_idx in range(N_PLAYERS):
    eng_df = compute_engagement_features(X[p_idx])
    start = p_idx * N_ENGAGEMENTS
    end = start + N_ENGAGEMENTS
    engagement_feature_matrix[start:end] = eng_df[feature_names].values
    player_idx_arr[start:end] = p_idx
    engagement_idx_arr[start:end] = np.arange(N_ENGAGEMENTS)

engagement_df = pd.DataFrame(engagement_feature_matrix, columns=feature_names)
engagement_df["player_idx"] = player_idx_arr
engagement_df["engagement_idx"] = engagement_idx_arr

# player-level label kept ONLY as a separate lookup -- never joined onto
# engagement_df, so it can't leak into the unsupervised fit
player_labels = pd.Series(y, index=np.arange(N_PLAYERS), name="label")

nan_counts_before = engagement_df[feature_names].isna().sum()
for col in feature_names:
    n_nan = int(nan_counts_before[col])
    if n_nan > 0:
        median_val = engagement_df[col].median()
        engagement_df[col] = engagement_df[col].fillna(median_val)
        print(f"filled {col}: {n_nan} NaN ({n_nan / len(engagement_df) * 100:.4f}%) with median={median_val:.4f}")

print()
print(f"Engagement-level feature matrix shape: {engagement_df[feature_names].shape}")
print(f"Full engagement_df shape (incl. ID columns): {engagement_df.shape}")
print(f"player_labels shape: {player_labels.shape}")
print(f"NaN count after fill: {int(engagement_df[feature_names].isna().sum().sum())}")

filled peak_yaw_delta: 38 NaN (0.0106%) with median=13.7820
filled mean_yaw_delta: 38 NaN (0.0106%) with median=0.8663
filled yaw_jerk: 38 NaN (0.0106%) with median=1.2355

Engagement-level feature matrix shape: (360000, 11)
Full engagement_df shape (incl. ID columns): (360000, 13)
player_labels shape: (12000,)
NaN count after fill: 0


In [23]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
iso_forest.fit(engagement_df[feature_names].values)  # unsupervised -- no y passed

raw_decision = iso_forest.decision_function(engagement_df[feature_names].values)
# decision_function: higher = more normal/inlier, lower (more negative) = more anomalous.
# Flip sign so higher = more anomalous, consistent with earlier convention.
anomaly_scores = -raw_decision

# normalise to 0-1
score_min, score_max = anomaly_scores.min(), anomaly_scores.max()
normalized_scores = (anomaly_scores - score_min) / (score_max - score_min)
engagement_df["anomaly_score"] = normalized_scores

print("Engagement anomaly score distribution (0=most normal, 1=most anomalous):")
print(f"  min:  {normalized_scores.min():.4f}")
print(f"  max:  {normalized_scores.max():.4f}")
print(f"  mean: {normalized_scores.mean():.4f}")
print(f"  std:  {normalized_scores.std():.4f}")
for p in [50, 90, 95, 99]:
    print(f"  p{p}:  {np.percentile(normalized_scores, p):.4f}")

Engagement anomaly score distribution (0=most normal, 1=most anomalous):
  min:  0.0000
  max:  1.0000
  mean: 0.1634
  std:  0.1299
  p50:  0.1211
  p90:  0.3481
  p95:  0.4439
  p99:  0.6109


In [24]:
# Approach A: mean engagement anomaly score per player
mean_score_per_player = (
    engagement_df.groupby("player_idx")["anomaly_score"].mean().reindex(range(N_PLAYERS))
)
auc_mean_approach = roc_auc_score(player_labels.values, mean_score_per_player.values)

# Approach B: count of flagged engagements per player, at two thresholds
threshold_90 = np.percentile(normalized_scores, 90)
threshold_95 = np.percentile(normalized_scores, 95)

engagement_df["flagged_90"] = (engagement_df["anomaly_score"] > threshold_90).astype(int)
engagement_df["flagged_95"] = (engagement_df["anomaly_score"] > threshold_95).astype(int)

count_flagged_90 = engagement_df.groupby("player_idx")["flagged_90"].sum().reindex(range(N_PLAYERS))
count_flagged_95 = engagement_df.groupby("player_idx")["flagged_95"].sum().reindex(range(N_PLAYERS))

auc_count_90 = roc_auc_score(player_labels.values, count_flagged_90.values)
auc_count_95 = roc_auc_score(player_labels.values, count_flagged_95.values)

print(f"Approach A -- mean engagement score per player:        AUC-ROC = {auc_mean_approach:.4f}")
print(f"Approach B -- count flagged engagements @ 90th pct:     AUC-ROC = {auc_count_90:.4f}")
print(f"Approach B -- count flagged engagements @ 95th pct:     AUC-ROC = {auc_count_95:.4f}")
print()

approach_aucs = {
    "mean_score": auc_mean_approach,
    "count_flagged_90": auc_count_90,
    "count_flagged_95": auc_count_95,
}
best_approach_name = max(approach_aucs, key=approach_aucs.get)
print(f"Highest: {best_approach_name} (AUC-ROC = {approach_aucs[best_approach_name]:.4f})")
print(f"11-feature player-level RF (Cell 16) for comparison:    AUC-ROC = {auc_rf:.4f}")
print(f"Delta (best engagement-level - player-level RF): {approach_aucs[best_approach_name] - auc_rf:+.4f}")

Approach A -- mean engagement score per player:        AUC-ROC = 0.5279
Approach B -- count flagged engagements @ 90th pct:     AUC-ROC = 0.5037
Approach B -- count flagged engagements @ 95th pct:     AUC-ROC = 0.5023

Highest: mean_score (AUC-ROC = 0.5279)
11-feature player-level RF (Cell 16) for comparison:    AUC-ROC = 0.6578
Delta (best engagement-level - player-level RF): -0.1300


In [25]:
from sklearn.metrics import precision_recall_curve

approach_scores = {
    "mean_score": mean_score_per_player.values,
    "count_flagged_90": count_flagged_90.values,
    "count_flagged_95": count_flagged_95.values,
}
best_scores = approach_scores[best_approach_name]

precisions, recalls, thresholds = precision_recall_curve(player_labels.values, best_scores)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-12)
best_idx = int(np.argmax(f1_scores))

best_threshold = thresholds[best_idx]
best_precision = precisions[best_idx]
best_recall = recalls[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Best approach: {best_approach_name} (AUC-ROC = {approach_aucs[best_approach_name]:.4f})")
print(f"Optimal threshold (maximizes F1): {best_threshold:.4f}")
print(f"  Precision (cheater class): {best_precision:.4f}")
print(f"  Recall    (cheater class): {best_recall:.4f}")
print(f"  F1        (cheater class): {best_f1:.4f}")

Best approach: mean_score (AUC-ROC = 0.5279)
Optimal threshold (maximizes F1): 0.0939
  Precision (cheater class): 0.1680
  Recall    (cheater class): 0.9660
  F1        (cheater class): 0.2862


In [26]:
# Novelty detection check: fit Isolation Forest ONLY on legit engagements
# (learn what "normal" looks like), then score every engagement -- including
# cheater engagements -- against that boundary. Reuses engagement_df and
# player_labels from Cell 22, no regeneration.

engagement_player_label = player_labels.values[engagement_df["player_idx"].values]
legit_mask = engagement_player_label == 0
cheater_mask = engagement_player_label == 1

legit_engagements = engagement_df[legit_mask]
cheater_engagements = engagement_df[cheater_mask]

print(f"legit engagements:   {len(legit_engagements)} (expected {10000 * 30})")
print(f"cheater engagements: {len(cheater_engagements)} (expected {2000 * 30})")

legit engagements:   300000 (expected 300000)
cheater engagements: 60000 (expected 60000)


In [27]:
iso_forest_novelty = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
iso_forest_novelty.fit(legit_engagements[feature_names].values)  # fit on legit ONLY

# score ALL 360000 engagements (legit + cheater) against the legit-only boundary
raw_decision_novelty = iso_forest_novelty.decision_function(engagement_df[feature_names].values)
anomaly_scores_novelty = -raw_decision_novelty  # higher = more anomalous

score_min_nov, score_max_nov = anomaly_scores_novelty.min(), anomaly_scores_novelty.max()
normalized_scores_novelty = (anomaly_scores_novelty - score_min_nov) / (score_max_nov - score_min_nov)
engagement_df["anomaly_score_novelty"] = normalized_scores_novelty

legit_scores_novelty = normalized_scores_novelty[legit_mask]
cheater_scores_novelty = normalized_scores_novelty[cheater_mask]


def print_score_stats(name, arr):
    print(f"{name} (n={len(arr)}):")
    print(f"  min={arr.min():.4f} max={arr.max():.4f} mean={arr.mean():.4f} std={arr.std():.4f}")
    print(f"  p50={np.percentile(arr, 50):.4f} p90={np.percentile(arr, 90):.4f} "
          f"p95={np.percentile(arr, 95):.4f} p99={np.percentile(arr, 99):.4f}")


print("Novelty-detection engagement anomaly scores (fit on legit only):")
print_score_stats("legit engagements", legit_scores_novelty)
print_score_stats("cheater engagements", cheater_scores_novelty)

Novelty-detection engagement anomaly scores (fit on legit only):
legit engagements (n=300000):
  min=0.0000 max=1.0000 mean=0.1603 std=0.1322
  p50=0.1169 p90=0.3464 p95=0.4414 p99=0.6139
cheater engagements (n=60000):
  min=0.0039 max=0.9852 mean=0.1665 std=0.1339
  p50=0.1236 p90=0.3539 p95=0.4474 p99=0.6243


In [28]:
# Approach A: mean novelty score per player
mean_score_per_player_novelty = (
    engagement_df.groupby("player_idx")["anomaly_score_novelty"].mean().reindex(range(N_PLAYERS))
)
auc_mean_novelty = roc_auc_score(player_labels.values, mean_score_per_player_novelty.values)

# Approach B: count of flagged engagements per player, at two thresholds
threshold_90_novelty = np.percentile(normalized_scores_novelty, 90)
threshold_95_novelty = np.percentile(normalized_scores_novelty, 95)

engagement_df["flagged_90_novelty"] = (engagement_df["anomaly_score_novelty"] > threshold_90_novelty).astype(int)
engagement_df["flagged_95_novelty"] = (engagement_df["anomaly_score_novelty"] > threshold_95_novelty).astype(int)

count_flagged_90_novelty = engagement_df.groupby("player_idx")["flagged_90_novelty"].sum().reindex(range(N_PLAYERS))
count_flagged_95_novelty = engagement_df.groupby("player_idx")["flagged_95_novelty"].sum().reindex(range(N_PLAYERS))

auc_count_90_novelty = roc_auc_score(player_labels.values, count_flagged_90_novelty.values)
auc_count_95_novelty = roc_auc_score(player_labels.values, count_flagged_95_novelty.values)

print("Novelty detection (legit-only fit) aggregation results:")
print(f"  mean score per player:        AUC-ROC = {auc_mean_novelty:.4f}")
print(f"  count flagged @ 90th pct:     AUC-ROC = {auc_count_90_novelty:.4f}")
print(f"  count flagged @ 95th pct:     AUC-ROC = {auc_count_95_novelty:.4f}")
print()
print("Mixed unsupervised fit (Cell 24) for comparison:")
print(f"  mean score per player:        AUC-ROC = {auc_mean_approach:.4f}")
print(f"  count flagged @ 90th pct:     AUC-ROC = {auc_count_90:.4f}")
print(f"  count flagged @ 95th pct:     AUC-ROC = {auc_count_95:.4f}")
print()
print(f"11-feature player-level RF (Cell 16) baseline: AUC-ROC = {auc_rf:.4f}")
print()

approach_aucs_novelty = {
    "mean_score_novelty": auc_mean_novelty,
    "count_flagged_90_novelty": auc_count_90_novelty,
    "count_flagged_95_novelty": auc_count_95_novelty,
}
best_approach_name_novelty = max(approach_aucs_novelty, key=approach_aucs_novelty.get)
best_auc_novelty = approach_aucs_novelty[best_approach_name_novelty]
print(f"Highest novelty-detection approach: {best_approach_name_novelty} (AUC-ROC = {best_auc_novelty:.4f})")
print(f"Delta vs best mixed-fit approach ({best_approach_name}, {approach_aucs[best_approach_name]:.4f}): "
      f"{best_auc_novelty - approach_aucs[best_approach_name]:+.4f}")
print(f"Delta vs player-level RF ({auc_rf:.4f}): {best_auc_novelty - auc_rf:+.4f}")

Novelty detection (legit-only fit) aggregation results:
  mean score per player:        AUC-ROC = 0.5322
  count flagged @ 90th pct:     AUC-ROC = 0.5115
  count flagged @ 95th pct:     AUC-ROC = 0.5083

Mixed unsupervised fit (Cell 24) for comparison:
  mean score per player:        AUC-ROC = 0.5279
  count flagged @ 90th pct:     AUC-ROC = 0.5037
  count flagged @ 95th pct:     AUC-ROC = 0.5023

11-feature player-level RF (Cell 16) baseline: AUC-ROC = 0.6578

Highest novelty-detection approach: mean_score_novelty (AUC-ROC = 0.5322)
Delta vs best mixed-fit approach (mean_score, 0.5279): +0.0044
Delta vs player-level RF (0.6578): -0.1256


In [29]:
approach_scores_novelty = {
    "mean_score_novelty": mean_score_per_player_novelty.values,
    "count_flagged_90_novelty": count_flagged_90_novelty.values,
    "count_flagged_95_novelty": count_flagged_95_novelty.values,
}
best_scores_novelty = approach_scores_novelty[best_approach_name_novelty]

precisions_nov, recalls_nov, thresholds_nov = precision_recall_curve(player_labels.values, best_scores_novelty)
f1_scores_nov = 2 * precisions_nov[:-1] * recalls_nov[:-1] / (precisions_nov[:-1] + recalls_nov[:-1] + 1e-12)
best_idx_nov = int(np.argmax(f1_scores_nov))

best_threshold_nov = thresholds_nov[best_idx_nov]
best_precision_nov = precisions_nov[best_idx_nov]
best_recall_nov = recalls_nov[best_idx_nov]
best_f1_nov = f1_scores_nov[best_idx_nov]

print(f"Best novelty-detection approach: {best_approach_name_novelty} (AUC-ROC = {best_auc_novelty:.4f})")
print(f"Optimal threshold (maximizes F1): {best_threshold_nov:.4f}")
print(f"  Precision (cheater class): {best_precision_nov:.4f}")
print(f"  Recall    (cheater class): {best_recall_nov:.4f}")
print(f"  F1        (cheater class): {best_f1_nov:.4f}")

Best novelty-detection approach: mean_score_novelty (AUC-ROC = 0.5322)
Optimal threshold (maximizes F1): 0.0911
  Precision (cheater class): 0.1685
  Recall    (cheater class): 0.9640
  F1        (cheater class): 0.2869


In [30]:
# Player-level unsupervised Isolation Forest (Doc 06 Phase 2.1's originally
# specified approach, not yet tested). Reuses the existing 80/20 stratified
# splits (Cell 14) and scaler (loaded for reference only -- IF is tree-based
# and doesn't need scaled input) rather than regenerating either.

splits_dir_if = Path("../data/splits")
X_train_if = np.load(splits_dir_if / "X_train.npy")
X_test_if = np.load(splits_dir_if / "X_test.npy")
y_train_if = np.load(splits_dir_if / "y_train.npy")
y_test_if = np.load(splits_dir_if / "y_test.npy")

scaler_loaded = joblib.load(Path("../trained_models/scaler.pkl"))

# Variant A: mixed fit -- all of X_train (cheater + legit together), unsupervised
iso_forest_player_mixed = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
iso_forest_player_mixed.fit(X_train_if)  # no y passed

raw_decision_mixed_player = iso_forest_player_mixed.decision_function(X_test_if)
anomaly_scores_mixed_player = -raw_decision_mixed_player
min_m, max_m = anomaly_scores_mixed_player.min(), anomaly_scores_mixed_player.max()
normalized_scores_mixed_player = (anomaly_scores_mixed_player - min_m) / (max_m - min_m)

# Variant B: legit-only novelty fit -- only y_train==0 rows, unsupervised
X_train_legit_only = X_train_if[y_train_if == 0]
iso_forest_player_novelty = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
iso_forest_player_novelty.fit(X_train_legit_only)

raw_decision_novelty_player = iso_forest_player_novelty.decision_function(X_test_if)
anomaly_scores_novelty_player = -raw_decision_novelty_player
min_n, max_n = anomaly_scores_novelty_player.min(), anomaly_scores_novelty_player.max()
normalized_scores_novelty_player = (anomaly_scores_novelty_player - min_n) / (max_n - min_n)

print(f"X_train_if shape: {X_train_if.shape}, X_test_if shape: {X_test_if.shape}")
print(f"X_train_legit_only shape (variant B fit set): {X_train_legit_only.shape}")
print(f"Variant A (mixed) test scores:   min={normalized_scores_mixed_player.min():.4f} "
      f"max={normalized_scores_mixed_player.max():.4f} mean={normalized_scores_mixed_player.mean():.4f}")
print(f"Variant B (legit-only) test scores: min={normalized_scores_novelty_player.min():.4f} "
      f"max={normalized_scores_novelty_player.max():.4f} mean={normalized_scores_novelty_player.mean():.4f}")

X_train_if shape: (9600, 11), X_test_if shape: (2400, 11)
X_train_legit_only shape (variant B fit set): (8000, 11)
Variant A (mixed) test scores:   min=0.0000 max=1.0000 mean=0.2216
Variant B (legit-only) test scores: min=0.0000 max=1.0000 mean=0.2183


In [31]:
auc_if_mixed_player = roc_auc_score(y_test_if, normalized_scores_mixed_player)
auc_if_novelty_player = roc_auc_score(y_test_if, normalized_scores_novelty_player)

print("Player-level unsupervised Isolation Forest:")
print(f"  Variant A -- mixed fit:       AUC-ROC = {auc_if_mixed_player:.4f}")
print(f"  Variant B -- legit-only fit:  AUC-ROC = {auc_if_novelty_player:.4f}")
print()
print("Prior results for comparison:")
print(f"  Engagement-level mixed fit (best agg):        AUC-ROC = 0.5279")
print(f"  Engagement-level legit-only novelty (best agg): AUC-ROC = 0.5322")
print(f"  Player-level supervised RF (Cell 16):           AUC-ROC = {auc_rf:.4f}")
print()

best_player_if_name = "mixed" if auc_if_mixed_player >= auc_if_novelty_player else "novelty"
best_player_if_auc = max(auc_if_mixed_player, auc_if_novelty_player)
print(f"Better player-level IF variant: {best_player_if_name} (AUC-ROC = {best_player_if_auc:.4f})")
print(f"Delta vs player-level supervised RF: {best_player_if_auc - auc_rf:+.4f}")

Player-level unsupervised Isolation Forest:
  Variant A -- mixed fit:       AUC-ROC = 0.5784
  Variant B -- legit-only fit:  AUC-ROC = 0.5866

Prior results for comparison:
  Engagement-level mixed fit (best agg):        AUC-ROC = 0.5279
  Engagement-level legit-only novelty (best agg): AUC-ROC = 0.5322
  Player-level supervised RF (Cell 16):           AUC-ROC = 0.6578

Better player-level IF variant: novelty (AUC-ROC = 0.5866)
Delta vs player-level supervised RF: -0.0712


In [32]:
from sklearn.metrics import confusion_matrix

best_player_if_scores = (
    normalized_scores_mixed_player if best_player_if_name == "mixed" else normalized_scores_novelty_player
)

precisions_p, recalls_p, thresholds_p = precision_recall_curve(y_test_if, best_player_if_scores)
f1_scores_p = 2 * precisions_p[:-1] * recalls_p[:-1] / (precisions_p[:-1] + recalls_p[:-1] + 1e-12)
best_idx_p = int(np.argmax(f1_scores_p))

best_threshold_p = thresholds_p[best_idx_p]
best_precision_p = precisions_p[best_idx_p]
best_recall_p = recalls_p[best_idx_p]
best_f1_p = f1_scores_p[best_idx_p]

y_pred_p = (best_player_if_scores >= best_threshold_p).astype(int)
cm_p = confusion_matrix(y_test_if, y_pred_p)

print(f"Best player-level IF variant: {best_player_if_name} (AUC-ROC = {best_player_if_auc:.4f})")
print(f"Optimal threshold (maximizes F1): {best_threshold_p:.4f}")
print(f"  Precision (cheater class): {best_precision_p:.4f}")
print(f"  Recall    (cheater class): {best_recall_p:.4f}")
print(f"  F1        (cheater class): {best_f1_p:.4f}")
print()
print("Confusion matrix (rows=true, cols=predicted, order=[legit, cheater]):")
print(cm_p)

Best player-level IF variant: novelty (AUC-ROC = 0.5866)
Optimal threshold (maximizes F1): 0.1198
  Precision (cheater class): 0.1904
  Recall    (cheater class): 0.8075
  F1        (cheater class): 0.3082

Confusion matrix (rows=true, cols=predicted, order=[legit, cheater]):
[[ 627 1373]
 [  77  323]]


In [33]:
TRUE_CHEATER_RATE = 0.1667

if best_player_if_name == "mixed":
    iso_forest_player_contam = IsolationForest(
        n_estimators=200, contamination=TRUE_CHEATER_RATE, random_state=42
    )
    iso_forest_player_contam.fit(X_train_if)
else:
    iso_forest_player_contam = IsolationForest(
        n_estimators=200, contamination=TRUE_CHEATER_RATE, random_state=42
    )
    iso_forest_player_contam.fit(X_train_legit_only)

raw_decision_contam = iso_forest_player_contam.decision_function(X_test_if)
anomaly_scores_contam = -raw_decision_contam
min_c, max_c = anomaly_scores_contam.min(), anomaly_scores_contam.max()
normalized_scores_contam = (anomaly_scores_contam - min_c) / (max_c - min_c)

auc_if_contam = roc_auc_score(y_test_if, normalized_scores_contam)

print(f"Player-level IF ({best_player_if_name} fit), contamination={TRUE_CHEATER_RATE}: "
      f"AUC-ROC = {auc_if_contam:.4f}")
print(f"Player-level IF ({best_player_if_name} fit), contamination='auto':          "
      f"AUC-ROC = {best_player_if_auc:.4f}")
print(f"Delta: {auc_if_contam - best_player_if_auc:+.4f}")

Player-level IF (novelty fit), contamination=0.1667: AUC-ROC = 0.5866
Player-level IF (novelty fit), contamination='auto':          AUC-ROC = 0.5866
Delta: +0.0000


In [34]:
# Cleanlab label noise check: does a meaningful fraction of the "cheater"/
# "legit" labels in the original dataset look wrong? If so, that could
# explain part of the ~0.58-0.66 AUC ceiling hit across many feature/model
# combinations so far (Cells 16, 20, 21, 25, 29, 33). Loads features_hybrid.csv
# fresh rather than reusing df_hybrid/hybrid_feature_cols already in memory,
# so this check is self-contained and reproducible on its own.

df_cleanlab = pd.read_csv(Path("../data/features_hybrid.csv"))

cleanlab_exclude_cols = [
    "peak_yaw_delta_source_engagement",
    "peak_yaw_delta_source_tick",
    "min_cv_yaw_source_engagement",
    "min_cv_yaw_source_tick",
    "peak_yaw_delta_real_time_seconds",
    "min_cv_yaw_real_time_seconds",
    "label",
]
cleanlab_feature_cols = [c for c in df_cleanlab.columns if c not in cleanlab_exclude_cols]

X_cleanlab = df_cleanlab[cleanlab_feature_cols].values
y_cleanlab = df_cleanlab["label"].values

print(f"df_cleanlab shape: {df_cleanlab.shape}")
print(f"feature columns ({len(cleanlab_feature_cols)}): {cleanlab_feature_cols}")
print(f"X_cleanlab shape: {X_cleanlab.shape}, y_cleanlab shape: {y_cleanlab.shape}")
print(f"cheater rate: {y_cleanlab.mean():.4f}")

df_cleanlab shape: (12000, 18)
feature columns (11): ['peak_yaw_delta', 'peak_pitch_delta', 'min_cv_yaw', 'min_cv_pitch', 'mean_yaw_delta', 'snap_count', 'cv_yaw_std', 'cv_pitch_std', 'fire_on_target_rate', 'yaw_jerk', 'engagement_firing_rate']
X_cleanlab shape: (12000, 11), y_cleanlab shape: (12000,)
cheater rate: 0.1667


In [35]:
from sklearn.model_selection import cross_val_predict

# Out-of-fold predicted probabilities -- cleanlab needs honest, held-out-style
# probabilities (not probabilities from a model that memorized the labels
# it was trained on) to detect label issues correctly.
rf_cleanlab = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
pred_probs_cleanlab = cross_val_predict(
    rf_cleanlab, X_cleanlab, y_cleanlab, cv=5, method="predict_proba"
)

print(f"pred_probs_cleanlab shape: {pred_probs_cleanlab.shape}")
print(f"mean predicted P(cheater): {pred_probs_cleanlab[:, 1].mean():.4f}")

pred_probs_cleanlab shape: (12000, 2)
mean predicted P(cheater): 0.2891


In [36]:
from cleanlab.filter import find_label_issues

label_issue_indices = find_label_issues(
    labels=y_cleanlab,
    pred_probs=pred_probs_cleanlab,
    return_indices_ranked_by="self_confidence",
)

n_flagged = len(label_issue_indices)
pct_flagged = 100 * n_flagged / len(y_cleanlab)

flagged_labels = y_cleanlab[label_issue_indices]
n_flagged_cheater = int((flagged_labels == 1).sum())
n_flagged_legit = int((flagged_labels == 0).sum())
pct_flagged_cheater = 100 * n_flagged_cheater / n_flagged if n_flagged else 0.0
pct_flagged_legit = 100 * n_flagged_legit / n_flagged if n_flagged else 0.0

print(f"Total flagged label issues: {n_flagged} / {len(y_cleanlab)} ({pct_flagged:.2f}%)")
print()
print("Breakdown of flagged issues by original label:")
print(f"  originally labeled cheater (1): {n_flagged_cheater} ({pct_flagged_cheater:.1f}% of flagged)")
print(f"  originally labeled legit   (0): {n_flagged_legit} ({pct_flagged_legit:.1f}% of flagged)")

Total flagged label issues: 1350 / 12000 (11.25%)

Breakdown of flagged issues by original label:
  originally labeled cheater (1): 855 (63.3% of flagged)
  originally labeled legit   (0): 495 (36.7% of flagged)


In [37]:
top10_issue_idx = label_issue_indices[:10]

top10_rows = []
for idx in top10_issue_idx:
    top10_rows.append({
        "row_index": int(idx),
        "original_label": "cheater" if y_cleanlab[idx] == 1 else "legit",
        "pred_prob_legit": pred_probs_cleanlab[idx, 0],
        "pred_prob_cheater": pred_probs_cleanlab[idx, 1],
    })

top10_issues_df = pd.DataFrame(top10_rows)
print("Top 10 most confident label issues (ranked by self-confidence):")
print(top10_issues_df.to_string(index=False))

Top 10 most confident label issues (ranked by self-confidence):
 row_index original_label  pred_prob_legit  pred_prob_cheater
      6331          legit            0.010              0.990
      6311          legit            0.015              0.985
      6371          legit            0.015              0.985
      6333          legit            0.015              0.985
      6372          legit            0.020              0.980
      6316          legit            0.020              0.980
      6332          legit            0.020              0.980
       414        cheater            0.975              0.025
      6329          legit            0.025              0.975
       667        cheater            0.975              0.025


In [38]:
cleanlab_clean_mask = np.ones(len(y_cleanlab), dtype=bool)
cleanlab_clean_mask[label_issue_indices] = False

X_cleaned = X_cleanlab[cleanlab_clean_mask]
y_cleaned = y_cleanlab[cleanlab_clean_mask]

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, stratify=y_cleaned, random_state=42
)

rf_cleaned = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_cleaned.fit(X_train_clean, y_train_clean)

y_proba_rf_cleaned = rf_cleaned.predict_proba(X_test_clean)[:, 1]
auc_rf_cleaned = roc_auc_score(y_test_clean, y_proba_rf_cleaned)

AUC_ORIGINAL_UNCLEANED = 0.6578  # Cell 16, Random Forest on full uncleaned hybrid features
delta_cleaned_vs_original = auc_rf_cleaned - AUC_ORIGINAL_UNCLEANED

print(f"Removed {n_flagged} flagged label-issue rows ({pct_flagged:.2f}% of data)")
print(f"Cleaned data shape: {X_cleaned.shape}")
print("Random Forest on cleaned data (fresh 80/20 stratified split, random_state=42):")
print(f"  AUC-ROC: {auc_rf_cleaned:.4f}")
print()
print(f"Original AUC-ROC (Cell 16, uncleaned, full hybrid features): {AUC_ORIGINAL_UNCLEANED:.4f}")
print(f"Delta (cleaned - original): {delta_cleaned_vs_original:+.4f}")

Removed 1350 flagged label-issue rows (11.25% of data)
Cleaned data shape: (10650, 11)
Random Forest on cleaned data (fresh 80/20 stratified split, random_state=42):
  AUC-ROC: 0.8820

Original AUC-ROC (Cell 16, uncleaned, full hybrid features): 0.6578
Delta (cleaned - original): +0.2242


In [39]:
# Fix for Cell 38's test-set contamination: Cell 38 re-split the CLEANED
# pool into a fresh train/test, so its test set also had hard/ambiguous
# rows removed -- that inflates AUC by evaluating only on easier remaining
# examples, not because cleaning genuinely improved generalization. The
# correct check is to clean ONLY the training rows and evaluate on the
# ORIGINAL, untouched test set (backend/data/splits/, from Cell 14).
#
# The saved split .npy files carry no row-index labels, so to know which
# of Cell 36's 1350 flagged (df_hybrid-row-order) indices fall inside the
# saved X_train, the same train_test_split(test_size=0.2, stratify=y_hybrid,
# random_state=42) call is replayed on an index array. StratifiedShuffleSplit
# partitions depend only on n_samples, the stratify labels, and random_state
# -- not on the feature values -- so replaying it on np.arange(12000) with
# the same y_hybrid/random_state reproduces the exact same partition Cell 14
# produced, and the assertions below confirm that against the saved arrays.

splits_dir_clean = Path("../data/splits")
X_train_orig = np.load(splits_dir_clean / "X_train.npy")
X_test_orig = np.load(splits_dir_clean / "X_test.npy")
y_train_orig = np.load(splits_dir_clean / "y_train.npy")
y_test_orig = np.load(splits_dir_clean / "y_test.npy")

idx_all = np.arange(len(y_hybrid))
idx_train, idx_test = train_test_split(
    idx_all, test_size=0.2, stratify=y_hybrid, random_state=42
)

assert np.allclose(X_hybrid[idx_train], X_train_orig), "train index reconstruction mismatch"
assert np.allclose(X_hybrid[idx_test], X_test_orig), "test index reconstruction mismatch"
assert np.array_equal(y_hybrid[idx_train], y_train_orig), "train label reconstruction mismatch"
assert np.array_equal(y_hybrid[idx_test], y_test_orig), "test label reconstruction mismatch"

flagged_set = set(int(i) for i in label_issue_indices)
train_flagged_mask = np.array([int(i) in flagged_set for i in idx_train])
n_train_flagged = int(train_flagged_mask.sum())

X_train_cleaned = X_train_orig[~train_flagged_mask]
y_train_cleaned = y_train_orig[~train_flagged_mask]

print(f"Flagged label issues total (Cell 36): {len(label_issue_indices)}")
print(f"Flagged issues falling in the ORIGINAL X_train split: {n_train_flagged} "
      f"({100 * n_train_flagged / len(label_issue_indices):.1f}% of all flagged)")
print()
print(f"Original X_train shape: {X_train_orig.shape}")
print(f"Cleaned X_train shape (flagged rows removed): {X_train_cleaned.shape}")
print(f"X_test shape (UNTOUCHED): {X_test_orig.shape}")

Flagged label issues total (Cell 36): 1350
Flagged issues falling in the ORIGINAL X_train split: 1068 (79.1% of all flagged)

Original X_train shape: (9600, 11)
Cleaned X_train shape (flagged rows removed): (8532, 11)
X_test shape (UNTOUCHED): (2400, 11)


In [40]:
rf_corrected = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_corrected.fit(X_train_cleaned, y_train_cleaned)

y_proba_rf_corrected = rf_corrected.predict_proba(X_test_orig)[:, 1]
auc_rf_corrected = roc_auc_score(y_test_orig, y_proba_rf_corrected)

AUC_ORIGINAL_BASELINE = 0.6578    # Cell 16: uncleaned train, uncleaned test
AUC_FLAWED_CLEANED_TEST = 0.8820  # Cell 38: cleaned train AND cleaned test (test-set contaminated)

delta_vs_original = auc_rf_corrected - AUC_ORIGINAL_BASELINE
delta_vs_flawed = auc_rf_corrected - AUC_FLAWED_CLEANED_TEST

print("Random Forest trained on CLEANED X_train, evaluated on ORIGINAL UNCLEANED X_test:")
print(f"  AUC-ROC: {auc_rf_corrected:.4f}")
print()
print(f"Original uncleaned baseline      (Cell 16, train+test both uncleaned): {AUC_ORIGINAL_BASELINE:.4f}")
print(f"Flawed cleaned-test result       (Cell 38, train+test both cleaned):   {AUC_FLAWED_CLEANED_TEST:.4f}")
print(f"Corrected result (this cell)     (train cleaned, test UNTOUCHED):     {auc_rf_corrected:.4f}")
print()
print(f"Honest delta vs original baseline:   {delta_vs_original:+.4f}")
print(f"Delta vs the flawed cleaned-test run: {delta_vs_flawed:+.4f}")

Random Forest trained on CLEANED X_train, evaluated on ORIGINAL UNCLEANED X_test:
  AUC-ROC: 0.6745

Original uncleaned baseline      (Cell 16, train+test both uncleaned): 0.6578
Flawed cleaned-test result       (Cell 38, train+test both cleaned):   0.8820
Corrected result (this cell)     (train cleaned, test UNTOUCHED):     0.6745

Honest delta vs original baseline:   +0.0167
Delta vs the flawed cleaned-test run: -0.2075


In [41]:
# Does cleanlab's flagged-issue list cluster near the cheaters/legit
# concatenation boundary (row 2000, where X = concatenate([cheaters, legit]))
# -- which would suggest a processing artifact -- or is it spread through
# the dataset, consistent with genuine scattered label noise?

flagged_arr = np.array(sorted(int(i) for i in label_issue_indices))
print(f"Flagged label issues: {len(flagged_arr)}")
print(f"min flagged index: {flagged_arr.min()}, max flagged index: {flagged_arr.max()}")
print()

bucket_edges = np.arange(0, 13000, 1000)
bucket_counts, _ = np.histogram(flagged_arr, bins=bucket_edges)
print("Flagged issue count per 1000-index bucket:")
for start, count in zip(bucket_edges[:-1], bucket_counts):
    print(f"  [{start:5d}, {start + 1000:5d}): {count}")
print()

boundary_mask = (flagged_arr >= 1900) & (flagged_arr < 2100)
n_boundary = int(boundary_mask.sum())
n_elsewhere = len(flagged_arr) - n_boundary
boundary_range_size = 200
expected_if_uniform = len(flagged_arr) * (boundary_range_size / 12000)

print(f"Flagged issues in [1900, 2100) (near cheaters/legit boundary at index 2000): {n_boundary}")
print(f"Flagged issues elsewhere: {n_elsewhere}")
print(f"Expected count in this 200-wide range if flags were uniformly spread: {expected_if_uniform:.1f}")
if expected_if_uniform > 0:
    print(f"Observed vs expected ratio: {n_boundary / expected_if_uniform:.2f}x")

Flagged label issues: 1350
min flagged index: 0, max flagged index: 11957

Flagged issue count per 1000-index bucket:
  [    0,  1000): 433
  [ 1000,  2000): 422
  [ 2000,  3000): 34
  [ 3000,  4000): 41
  [ 4000,  5000): 49
  [ 5000,  6000): 58
  [ 6000,  7000): 93
  [ 7000,  8000): 45
  [ 8000,  9000): 42
  [ 9000, 10000): 49
  [10000, 11000): 54
  [11000, 12000): 30

Flagged issues in [1900, 2100) (near cheaters/legit boundary at index 2000): 39
Flagged issues elsewhere: 1311
Expected count in this 200-wide range if flags were uniformly spread: 22.5
Observed vs expected ratio: 1.73x


In [42]:
def compute_new_features(player_tensor):
    """player_tensor: (30, 192, 5) -> dict of 4 new player-level features
    (never-tried aggregations/signals, not new aggregations of the existing
    11) plus lock_achieved_rate, computed directly from the raw tensor."""
    yaw = player_tensor[..., 0]
    pitch = player_tensor[..., 1]
    cv_yaw = player_tensor[..., 2]
    cv_pitch = player_tensor[..., 3]
    firing = player_tensor[..., 4]

    n_engagements, n_ticks = yaw.shape

    # a) yaw_pitch_combined_peak: combined 2D per-tick movement magnitude,
    # maxed within each engagement, then maxed across all 30 engagements.
    combined_mag = np.sqrt(yaw ** 2 + pitch ** 2)  # (30, 192)
    peak_combined_per_eng = np.max(combined_mag, axis=1)  # (30,)
    yaw_pitch_combined_peak = np.max(peak_combined_per_eng)

    # b) time_to_lock_ticks / lock_achieved_rate: per engagement, tick gap
    # from first off-target tick (|CrosshairToVictimYaw| > 20) to the next
    # locked-on tick (|CrosshairToVictimYaw| < 5) after it.
    tick_gaps = np.full(n_engagements, np.nan)
    for e in range(n_engagements):
        abs_cv_yaw_e = np.abs(cv_yaw[e])
        off_target_ticks = np.where(abs_cv_yaw_e > 20)[0]
        if len(off_target_ticks) == 0:
            continue
        first_off = off_target_ticks[0]
        locked_ticks = np.where(abs_cv_yaw_e[first_off + 1:] < 5)[0]
        if len(locked_ticks) == 0:
            continue
        first_lock = locked_ticks[0] + first_off + 1
        tick_gaps[e] = first_lock - first_off

    lock_achieved_rate = np.mean(~np.isnan(tick_gaps))
    time_to_lock_ticks = np.nanmean(tick_gaps)

    # c) yaw_entropy_min: Shannon entropy of the AttackerDeltaYaw tick-to-tick
    # distribution per engagement (20-bin histogram), MIN across engagements
    # -- the single most predictable/repetitive engagement for this player.
    entropies = np.full(n_engagements, np.nan)
    for e in range(n_engagements):
        # a handful of engagements (37/12000 players) retain leading-tick
        # NaNs in yaw after Cell 2's forward-fill (no prior tick to fill
        # from) -- drop those ticks before histogramming this engagement.
        yaw_e_finite = yaw[e][np.isfinite(yaw[e])]
        if len(yaw_e_finite) == 0:
            continue
        hist, _ = np.histogram(yaw_e_finite, bins=20)
        total = hist.sum()
        if total == 0:
            continue
        probs = hist / total
        entropies[e] = stats.entropy(probs[probs > 0])
    yaw_entropy_min = np.nanmin(entropies)

    # d) avg_shot_distance_min: mean crosshair-to-victim distance at firing
    # ticks only, per engagement, MIN across engagements -- their single
    # most accurate engagement.
    shot_distances = np.full(n_engagements, np.nan)
    for e in range(n_engagements):
        firing_mask = firing[e] == 1
        if not np.any(firing_mask):
            continue
        dist = np.sqrt(cv_yaw[e, firing_mask] ** 2 + cv_pitch[e, firing_mask] ** 2)
        shot_distances[e] = np.mean(dist)
    avg_shot_distance_min = np.nanmin(shot_distances)

    return {
        "yaw_pitch_combined_peak": yaw_pitch_combined_peak,
        "time_to_lock_ticks": time_to_lock_ticks,
        "lock_achieved_rate": lock_achieved_rate,
        "yaw_entropy_min": yaw_entropy_min,
        "avg_shot_distance_min": avg_shot_distance_min,
    }


# sanity check on a single player
test_new_features = compute_new_features(X[0])
print("compute_new_features sanity check (player 0):")
for k, v in test_new_features.items():
    print(f"  {k}: {v}")

compute_new_features sanity check (player 0):
  yaw_pitch_combined_peak: 40.45061492919922
  time_to_lock_ticks: 79.13636363636364
  lock_achieved_rate: 0.7333333333333333
  yaw_entropy_min: 0.7132354892841797
  avg_shot_distance_min: 0.2850140631198883


In [43]:
new_feature_rows = [compute_new_features(X[i]) for i in range(X.shape[0])]
df_new_features = pd.DataFrame(new_feature_rows)
df_new_features["label"] = y

new_feature_cols = [
    "yaw_pitch_combined_peak",
    "time_to_lock_ticks",
    "lock_achieved_rate",
    "yaw_entropy_min",
    "avg_shot_distance_min",
]

print(f"df_new_features shape: {df_new_features.shape}")
print()
print(df_new_features.describe())
print()
print("NaN count per column (before fill):")
nan_counts_new = df_new_features[new_feature_cols].isna().sum()
print(nan_counts_new)

for col in new_feature_cols:
    n_nan = int(nan_counts_new[col])
    if n_nan > 0:
        median_val = df_new_features[col].median()
        df_new_features[col] = df_new_features[col].fillna(median_val)
        print(f"  filled {col}: {n_nan} NaN ({n_nan / len(df_new_features) * 100:.2f}%) with median={median_val:.4f}")

print()
print("NaN count per column (after fill):")
print(df_new_features[new_feature_cols].isna().sum())

df_new_features shape: (12000, 6)

       yaw_pitch_combined_peak  time_to_lock_ticks  lock_achieved_rate  \
count             11963.000000        12000.000000        12000.000000   
mean                 42.012608           82.828168            0.847839   
std                  17.582714           15.796295            0.120723   
min                   5.866450           17.500000            0.200000   
25%                  29.843933           72.036398            0.766667   
50%                  37.953644           82.892857            0.866667   
75%                  49.362617           93.666667            0.933333   
max                 132.795776          138.157895            1.000000   

       yaw_entropy_min  avg_shot_distance_min         label  
count     12000.000000           12000.000000  12000.000000  
mean          0.586075               0.922223      0.166667  
std           0.235569               0.706922      0.372694  
min           0.000000               0.005099     

In [44]:
cheater_new = df_new_features[df_new_features["label"] == 1]
legit_new = df_new_features[df_new_features["label"] == 0]

mw_new_rows = []
for col in new_feature_cols:
    u_stat, p_val = stats.mannwhitneyu(
        cheater_new[col], legit_new[col], alternative="two-sided"
    )
    mw_new_rows.append({
        "feature_name": col,
        "u_statistic": u_stat,
        "p_value": p_val,
        "significant": p_val < ALPHA,
    })

mw_new_table = pd.DataFrame(mw_new_rows).sort_values("p_value", ascending=True).reset_index(drop=True)

print(f"Mann-Whitney U test on the 5 new features, alpha = {ALPHA}:")
print(mw_new_table.to_string(index=False))
print()
n_significant_new = int(mw_new_table["significant"].sum())
print(f"{n_significant_new} / {len(mw_new_table)} new features have p < {ALPHA}")

Mann-Whitney U test on the 5 new features, alpha = 0.01:
           feature_name  u_statistic  p_value  significant
        yaw_entropy_min   10630005.0 0.000008         True
yaw_pitch_combined_peak    9548890.0 0.001424         True
  avg_shot_distance_min    9648909.5 0.013047        False
     lock_achieved_rate   10243795.0 0.083336        False
     time_to_lock_ticks   10039365.5 0.780751        False

2 / 5 new features have p < 0.01


In [45]:
# Merge the 5 new features with the existing 11 hybrid features (same
# df_hybrid row order as X/y) into a 16-feature combined set, then apply the
# SAME cleaned-train-row removal (Cell 39's train_flagged_mask/idx_train)
# and the SAME original untouched test split (idx_test / X_test_orig /
# y_test_orig) for a fair apples-to-apples comparison against 0.6745.

df_combined_16 = df_hybrid[hybrid_feature_cols].copy()
for col in new_feature_cols:
    df_combined_16[col] = df_new_features[col].values

combined_16_feature_cols = hybrid_feature_cols + new_feature_cols

X_16 = df_combined_16[combined_16_feature_cols].values
y_16 = df_hybrid["label"].values

assert np.array_equal(y_16, y_hybrid)
assert np.allclose(X_16[:, :len(hybrid_feature_cols)], X_hybrid)

X_train_16_orig = X_16[idx_train]
X_test_16_orig = X_16[idx_test]
y_train_16_orig = y_16[idx_train]
y_test_16_orig = y_16[idx_test]

assert np.array_equal(y_train_16_orig, y_train_orig)
assert np.array_equal(y_test_16_orig, y_test_orig)
assert np.allclose(X_train_16_orig[:, :len(hybrid_feature_cols)], X_train_orig)
assert np.allclose(X_test_16_orig[:, :len(hybrid_feature_cols)], X_test_orig)

X_train_16_cleaned = X_train_16_orig[~train_flagged_mask]
y_train_16_cleaned = y_train_16_orig[~train_flagged_mask]

rf_16 = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_16.fit(X_train_16_cleaned, y_train_16_cleaned)

y_proba_rf_16 = rf_16.predict_proba(X_test_16_orig)[:, 1]
auc_rf_16 = roc_auc_score(y_test_16_orig, y_proba_rf_16)

AUC_CLEANED_11 = 0.6745          # Cell 40: 11 hybrid features, cleaned train / untouched test
AUC_ORIGINAL_UNCLEANED = 0.6578  # Cell 16: 11 hybrid features, uncleaned train+test

delta_vs_11 = auc_rf_16 - AUC_CLEANED_11
delta_vs_original = auc_rf_16 - AUC_ORIGINAL_UNCLEANED

print(f"16-feature combined set (11 hybrid + 5 new), cleaned train / untouched test:")
print(f"  X_train_16_cleaned shape: {X_train_16_cleaned.shape}")
print(f"  X_test_16_orig shape:     {X_test_16_orig.shape}")
print(f"  AUC-ROC: {auc_rf_16:.4f}")
print()
print(f"11-feature cleaned result    (Cell 40): {AUC_CLEANED_11:.4f}")
print(f"Original uncleaned baseline  (Cell 16): {AUC_ORIGINAL_UNCLEANED:.4f}")
print()
print(f"Delta vs 11-feature cleaned:   {delta_vs_11:+.4f}")
print(f"Delta vs original uncleaned:   {delta_vs_original:+.4f}")

16-feature combined set (11 hybrid + 5 new), cleaned train / untouched test:
  X_train_16_cleaned shape: (8532, 16)
  X_test_16_orig shape:     (2400, 16)
  AUC-ROC: 0.6670

11-feature cleaned result    (Cell 40): 0.6745
Original uncleaned baseline  (Cell 16): 0.6578

Delta vs 11-feature cleaned:   -0.0075
Delta vs original uncleaned:   +0.0092


In [46]:
# Train on the 5 NEW features ALONE (no old ones) -- checks whether they
# carry independent signal even if correlation with the old 11 hides their
# contribution in the combined model's feature importances.
n_hybrid = len(hybrid_feature_cols)
X_train_5_cleaned = X_train_16_cleaned[:, n_hybrid:]
X_test_5_orig = X_test_16_orig[:, n_hybrid:]

rf_5 = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_5.fit(X_train_5_cleaned, y_train_16_cleaned)

y_proba_rf_5 = rf_5.predict_proba(X_test_5_orig)[:, 1]
auc_rf_5 = roc_auc_score(y_test_16_orig, y_proba_rf_5)

print("Random Forest on the 5 NEW features ALONE (cleaned train / untouched test):")
print(f"  AUC-ROC: {auc_rf_5:.4f}")
print()
print(f"For reference:")
print(f"  11-feature cleaned    (Cell 40): {AUC_CLEANED_11:.4f}")
print(f"  16-feature combined   (Cell 45): {auc_rf_16:.4f}")
print(f"  5 new features alone  (this cell): {auc_rf_5:.4f}")

Random Forest on the 5 NEW features ALONE (cleaned train / untouched test):
  AUC-ROC: 0.5640

For reference:
  11-feature cleaned    (Cell 40): 0.6745
  16-feature combined   (Cell 45): 0.6670
  5 new features alone  (this cell): 0.5640


In [47]:
importance_16_df = pd.DataFrame({
    "feature_name": combined_16_feature_cols,
    "importance": rf_16.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
importance_16_df["rank"] = importance_16_df.index + 1

print("Feature importances, 16-feature combined Random Forest (Cell 45), descending:")
print(importance_16_df.to_string(index=False))
print()

new_feature_ranks = importance_16_df[importance_16_df["feature_name"].isin(new_feature_cols)]
print("Ranking of the 5 NEW features within the full 16:")
print(new_feature_ranks.to_string(index=False))

Feature importances, 16-feature combined Random Forest (Cell 45), descending:
           feature_name  importance  rank
           min_cv_pitch    0.146276     1
 engagement_firing_rate    0.111537     2
    fire_on_target_rate    0.092321     3
           cv_pitch_std    0.074464     4
         mean_yaw_delta    0.054856     5
               yaw_jerk    0.054185     6
       peak_pitch_delta    0.052385     7
             cv_yaw_std    0.050142     8
        yaw_entropy_min    0.049765     9
             snap_count    0.049638    10
  avg_shot_distance_min    0.049113    11
         peak_yaw_delta    0.048745    12
yaw_pitch_combined_peak    0.047802    13
     time_to_lock_ticks    0.045982    14
             min_cv_yaw    0.043997    15
     lock_achieved_rate    0.028790    16

Ranking of the 5 NEW features within the full 16:
           feature_name  importance  rank
        yaw_entropy_min    0.049765     9
  avg_shot_distance_min    0.049113    11
yaw_pitch_combined_peak    0.04